# Module 00: Why Agentic AI for Security?

**Estimated time: 30 minutes**

> *"Alert fatigue is a solvable problem. Agentic AI is one of the best tools we have."*

---

## Learning Objectives

By the end of this notebook, you will be able to:
- Explain why rule-based triage systems fail at scale
- Distinguish between a simple LLM call and an agentic system
- Run the SecurityTriageAI demo and interpret every panel of output
- Articulate the AI-as-augmentation mental model for security operations

## The Problem: Expert Judgment at Machine Speed

The average enterprise SOC receives **thousands of security alerts per day.**

| Approach | Speed | Accuracy | Scales? |
|---|---|---|---|
| Rule-based SIEM filtering | Fast | Low (misses novel attacks) | Yes, but degrades |
| Manual analyst triage | Slow | High | No (burnout) |
| MDR outsourcing | Medium | Medium | Expensive |
| **Agentic AI triage** | **Fast** | **Medium-High (improving)** | **Yes** |

The core tension: you need **expert-level judgment at machine speed**.

Agentic AI is not a replacement for analysts. It's a tireless junior analyst
who processes everything and surfaces what matters — so senior analysts can
focus their judgment where it counts.

## What Makes a System 'Agentic'?

**A simple LLM call:**
```
Alert text --> [LLM] --> Triage decision
```
The LLM only knows what you put in the prompt. It can't look things up.
Missing context means wrong answers or hallucination.

**An agentic system:**
```
Alert text
  --> [Agent thinks: 'I need MITRE context']
  --> [Agent calls mitre_lookup('encoded powershell')]
  --> [Observes: T1059.001, execution stage]
  --> [Agent thinks: 'Plus process chain from Office app... malicious']
  --> [Grounded triage decision with explainable reasoning]
```

Key differences:
- The agent **uses tools** to gather information it needs mid-reasoning
- The agent **reasons in steps** before concluding
- The agent can **course-correct** if initial interpretation seems incomplete

This is the **ReAct pattern** (Reason + Act). We dig into the implementation in Module 01.

In [ ]:
# Setup check -- run this cell first
import sys
import os
import json

# Add project root to path (assumes notebook is in notebooks/)
sys.path.insert(0, os.path.abspath('..'))

try:
    from src.pipeline.mock import MOCK_DECISIONS
    print('Setup OK -- found ' + str(len(MOCK_DECISIONS)) + ' sample alerts ready to triage')
except ImportError as e:
    print('Error: ' + str(e))
    print('Run from repo root: pip install -r requirements.txt')

In [ ]:
# Load and inspect a raw security alert -- this is the input
with open('../data/sample_alerts.json') as f:
    alerts = json.load(f)

alert = alerts[0]
print('=== RAW SECURITY ALERT ===')
print(json.dumps(alert, indent=2))

In [ ]:
# Now look at what the AI produces for the same alert
from src.pipeline.mock import MOCK_DECISIONS

decision = MOCK_DECISIONS[alert['id']]

print('=== AI TRIAGE DECISION ===')
print('Severity:   ' + decision['severity'] + '  (confidence: ' + str(int(decision['confidence']*100)) + '%)')
print('Escalate:   ' + str(decision['escalation_required']))
print('Kill Chain: ' + decision['kill_chain_phase'])
print()
print('MITRE ATT&CK Techniques:')
for t in decision['mitre_techniques']:
    print('  - ' + t)
print()
print('Reasoning:')
print(decision['reasoning'])
print()
print('Recommended Actions:')
for action in decision['recommended_actions']:
    print('  -> ' + action)

## What Just Happened

Compare the raw alert to the triage decision:

| Raw Alert | AI Triage Decision |
|---|---|
| Event data from one endpoint tool | Structured severity + confidence |
| No context about what this means | MITRE technique + kill chain phase |
| No action guidance | Specific, ordered recommended actions |
| No explanation | Full reasoning chain |

That transformation -- from raw event to analyst-ready decision -- is what the agent does.

**The reasoning chain is the key.** It's not just *what* the AI decided, but *why*.
An analyst can audit it, agree or disagree, and learn from it.

The AI doesn't replace analyst judgment. It structures the work so judgment happens faster.

## Discussion Questions

Think through these before moving to Module 01:

1. **The 80% problem**: The system gets ~80% accuracy vs. expert baselines in mock mode.
   For which use cases is 80% good enough? For which is it dangerous?

2. **The transparency question**: The AI shows its full reasoning chain.
   How does that change how you'd use it compared to a black-box severity score?

3. **The novel attack problem**: MITRE ATT&CK has ~200 techniques.
   What happens when an attacker uses a method that doesn't exist in MITRE yet?

---

## Module 00 Complete

You've seen:
- The gap between raw alert data and analyst-ready decisions
- What makes a system 'agentic' vs. a simple LLM call
- Both the input (raw alert) and output (triage decision) of the pipeline

**Next: [Module 01 -- ReAct Agents](01_react_agent.ipynb)**
Build the reasoning loop yourself, step by step.